## Entendendo o negócio

Uma grande loja online busca aumentar sua receita e, em parceria com o departamento de marketing, reuniu hipóteses de mudanças que podem contribuir para esse objetivo. Como os recursos e o tempo disponíveis são limitados, é necessário priorizar as hipóteses com maior potencial de impacto.

Neste projeto, as hipóteses serão avaliadas e priorizadas, e as mais promissoras serão testadas por meio de um experimento A/B. Em seguida, os resultados serão analisados para verificar se as mudanças produziram efeitos relevantes e embasar uma recomendação para o negócio.

# 1. Bibliotecas Usadas

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go # outros recursos gráficos da biblioteca plotly
from scipy import stats as st

# 2. Preparando os dados

## 2.1. Otimizando os dados:

### - <b>carregando uma amostra dos dados para otimiza-los:</b>

In [2]:
hipoteses_teste = pd.read_csv('../datasets/hypotheses_us.csv',sep = ';')
orders_teste = pd.read_csv('../datasets/orders_us.csv', nrows = 100)
visits_teste = pd.read_csv('../datasets/visits_us.csv')

### - <b>imprimindo 5 linhas de cada amostra:</b>

In [3]:
hipoteses_teste.sample(5)

,Hypothesis,Reach,Impact,Confidence,Effort
8,Launch a promotion that gives users discounts ...,1,9,9,5
1,Launch your own delivery service. This will sh...,2,5,4,10
5,Add a customer review page. This will increase...,3,2,2,3
3,Change the category structure. This will incre...,8,3,3,8
7,Add a subscription form to all the main pages....,10,7,8,5


In [4]:
orders_teste.sample(5)

,transactionId,visitorId,date,revenue,group
29,2220299125,3803269165,2019-08-15,15.8,A
28,2632372083,3591136796,2019-08-15,85.2,B
44,376170286,1294108251,2019-08-15,50.2,A
2,2961555356,4069496402,2019-08-15,10.2,A
32,3666913472,2038680547,2019-08-15,30.8,B


In [5]:
visits_teste.sample(5)

,date,group,visits
51,2019-08-21,B,656
53,2019-08-23,B,546
40,2019-08-10,B,369
8,2019-08-09,A,617
56,2019-08-26,B,676


Ops! Precisamos consertar os titulos das colunas de hipoteses_teste

### - <b>informações gerais das amostras:</b>

In [6]:
hipoteses_teste.info(memory_usage = 'deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Hypothesis  9 non-null      object
 1   Reach       9 non-null      int64 
 2   Impact      9 non-null      int64 
 3   Confidence  9 non-null      int64 
 4   Effort      9 non-null      int64 
dtypes: int64(4), object(1)
memory usage: 1.6 KB


In [7]:
orders_teste.info(memory_usage = 'deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   transactionId  100 non-null    int64  
 1   visitorId      100 non-null    int64  
 2   date           100 non-null    object 
 3   revenue        100 non-null    float64
 4   group          100 non-null    object 
dtypes: float64(1), int64(2), object(2)
memory usage: 13.1 KB


In [8]:
visits_teste.info(memory_usage = 'deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62 entries, 0 to 61
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   date    62 non-null     object
 1   group   62 non-null     object
 2   visits  62 non-null     int64 
dtypes: int64(1), object(2)
memory usage: 7.2 KB


In [9]:
# Verificando se 'group' é do tipo categorica

print(visits_teste['group'].unique())

print(orders_teste['group'].unique())

['A' 'B']
['B' 'A']


precisamos passar 'date' para datetime, e 'group' para category, em orders e visits.

### - <b> tratando os titulos das colunas de hipoteses_teste </b>

In [10]:
# Construindo uma função para tratar as colunas:
def trata_colunas(df):
    '''
        Essa função recebe um dataframe e retorna os titulos de suas colunas tratatos no formato SnakeCase, como uma lista.
        OBS: Não se aplica à caracteres especiais do latim.
    '''
    cols = df.columns
    cols_tratadas = []
    for col in cols:
        col = col.lower()
        col = col.replace(' ','_')
        cols_tratadas.append(col)
    cols = cols_tratadas
    return cols

In [11]:
hipoteses_teste.columns = trata_colunas(hipoteses_teste)

print(hipoteses_teste.columns)

Index(['hypothesis', 'reach', 'impact', 'confidence', 'effort'], dtype='object')


### - <b> tratando os tipos de dados das colunas dos dataframes orders e visits </b>

In [12]:
# orders_teste:
orders_teste['date'] = pd.to_datetime(orders_teste['date'], format = '%Y-%m-%d')
orders_teste['group'] = orders_teste['group'].astype('category')

# orders_teste:
visits_teste['date'] = pd.to_datetime(visits_teste['date'], format = '%Y-%m-%d')
visits_teste['group'] = visits_teste['group'].astype('category')

### - <b>verificando o resultado da otimização:</b>

In [13]:
orders_teste.info(memory_usage = 'deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   transactionId  100 non-null    int64         
 1   visitorId      100 non-null    int64         
 2   date           100 non-null    datetime64[ns]
 3   revenue        100 non-null    float64       
 4   group          100 non-null    category      
dtypes: category(1), datetime64[ns](1), float64(1), int64(2)
memory usage: 3.6 KB


In [14]:
visits_teste.info(memory_usage = 'deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62 entries, 0 to 61
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    62 non-null     datetime64[ns]
 1   group   62 non-null     category      
 2   visits  62 non-null     int64         
dtypes: category(1), datetime64[ns](1), int64(1)
memory usage: 1.4 KB


conseguimos otimizar a memória usada em orders em cerca de 4 vezes, e em visits em 5 vezes. Isso é ótimo. Vamos usar isso para otimizar nossos dados completos a seguir

## 2.2. Carregando os arquivos de dados completos:

In [16]:
# Usando os parâmetros dtype e parse_dates para carregar nossos dados completos:
# parse_dates recebe uma lista para tratar dados no formato datetime
# dtype recebe um dicionário para tratar os outros tipos de dados

hipoteses =  pd.read_csv('../datasets/hypotheses_us.csv',sep = ';') # esse não precisa

orders = pd.read_csv('../datasets/orders_us.csv', 
                     dtype={'group':'category'},
                     parse_dates=['date'])

visits = pd.read_csv('../datasets/visits_us.csv', 
                     dtype={'group':'category'}, 
                     parse_dates=['date'])


In [17]:
# Tratando os titulos das colunas de hipoteses e orders:
hipoteses.columns = trata_colunas(hipoteses)

orders.columns = trata_colunas(orders)
orders = orders.rename(columns = {'visitorid': 'visitor_id', 'transactionid': 'transaction_id'})

## 2.3. Verificando as primeiras linhas dos dados completos:

In [18]:
hipoteses.head()

,hypothesis,reach,impact,confidence,effort
0,Add two new channels for attracting traffic. T...,3,10,8,6
1,Launch your own delivery service. This will sh...,2,5,4,10
2,Add product recommendation blocks to the store...,8,3,7,3
3,Change the category structure. This will incre...,8,3,3,8
4,Change the background color on the main page. ...,3,1,1,1


In [19]:
orders.head()

,transaction_id,visitor_id,date,revenue,group
0,3667963787,3312258926,2019-08-15,30.4,B
1,2804400009,3642806036,2019-08-15,15.2,B
2,2961555356,4069496402,2019-08-15,10.2,A
3,3797467345,1196621759,2019-08-15,155.1,B
4,2282983706,2322279887,2019-08-15,40.5,B


In [20]:
visits.head()

,date,group,visits
0,2019-08-01,A,719
1,2019-08-02,A,619
2,2019-08-03,A,507
3,2019-08-04,A,717
4,2019-08-05,A,756


## 2.4. Informações gerais dos dados completos:

In [21]:
hipoteses.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   hypothesis  9 non-null      object
 1   reach       9 non-null      int64 
 2   impact      9 non-null      int64 
 3   confidence  9 non-null      int64 
 4   effort      9 non-null      int64 
dtypes: int64(4), object(1)
memory usage: 492.0+ bytes


In [22]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1197 entries, 0 to 1196
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   transaction_id  1197 non-null   int64         
 1   visitor_id      1197 non-null   int64         
 2   date            1197 non-null   datetime64[ns]
 3   revenue         1197 non-null   float64       
 4   group           1197 non-null   category      
dtypes: category(1), datetime64[ns](1), float64(1), int64(2)
memory usage: 38.8 KB


In [23]:
visits.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62 entries, 0 to 61
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    62 non-null     datetime64[ns]
 1   group   62 non-null     category      
 2   visits  62 non-null     int64         
dtypes: category(1), datetime64[ns](1), int64(1)
memory usage: 1.2 KB


## 2.5. Tratando outros tipos de erros nos conjuntos de dados (devemos nos livrar desses erros caso existam):

### - <b>Erro 1: Vamos verificar se existem usuários que realizaram pedidos, e estão em multiplos grupos (A e B ao mesmo tempo): </b>

Esse tipo de erro pode influenciar negativamente o teste: Não sabermos qual hipotese fez o usuario comprar mais, por exemplo

In [24]:
orders.head()

,transaction_id,visitor_id,date,revenue,group
0,3667963787,3312258926,2019-08-15,30.4,B
1,2804400009,3642806036,2019-08-15,15.2,B
2,2961555356,4069496402,2019-08-15,10.2,A
3,3797467345,1196621759,2019-08-15,155.1,B
4,2282983706,2322279887,2019-08-15,40.5,B


In [25]:
# Filtrando os usuários dos grupos A e B:
users_A = orders[orders['group'] == 'A']['visitor_id']
users_B = orders[orders['group'] == 'B']['visitor_id']

# Filtrando os usuarios que se repetem nos 2 grupos:
mask = (orders['visitor_id'].isin(users_A)) & (orders['visitor_id'].isin(users_B))
intersecao = orders[mask]['visitor_id']

print(f'Existem {len(intersecao.to_list())} usuários problemáticos. Devemos remove-los do nosso conjunto de dados. \nJá que não dar pra saber a qual grupo eles pertencem exclusivamente')

Existem 181 usuários problemáticos. Devemos remove-los do nosso conjunto de dados. 
Já que não dar pra saber a qual grupo eles pertencem exclusivamente


In [26]:
# removendo os usuários problemáticos:
mask_2 = ~orders['visitor_id'].isin(intersecao)
orders = orders[mask_2].reset_index(drop=True)
orders.head()

,transaction_id,visitor_id,date,revenue,group
0,3667963787,3312258926,2019-08-15,30.4,B
1,2804400009,3642806036,2019-08-15,15.2,B
2,3797467345,1196621759,2019-08-15,155.1,B
3,2282983706,2322279887,2019-08-15,40.5,B
4,182168103,935554773,2019-08-15,35.0,B


### - <b>Erro 2: Vamos ver se existem dados de pedidos duplicados </b>

Isso é um problema pois pode inflacionar nossos indicadores de teste

In [27]:
print(f'Uma ótima notícia: Existem {orders.duplicated().sum()} linhas duplicadas')

Uma ótima notícia: Existem 0 linhas duplicadas


### - <b>Erro 3: Vamos ver se as datas dos pedidos não saem fora do intervalo das datas das visitas: Pedidos antes da primeira data de visita, ou pedidos muito tempo depois da ultima data de visita</b>

Para comprar, primeiro o usuário tem que visitar

In [28]:
intervalo_visitas = [visits['date'].min(), visits['date'].max()]

intervalo_orders = [orders['date'].min(), orders['date'].max()]

print(f' Intervalo das visitas: {intervalo_visitas} \n Intervalo das vendas: {intervalo_orders}')

 Intervalo das visitas: [Timestamp('2019-08-01 00:00:00'), Timestamp('2019-08-31 00:00:00')] 
 Intervalo das vendas: [Timestamp('2019-08-01 00:00:00'), Timestamp('2019-08-31 00:00:00')]


Tudo ok! Não temos nenhum erro: Os intervalos coincidem

### - <b>Erro 4: Vamos detectar se existem pedidos com valores irreais (negativos)</b>

In [29]:
receita_minima = orders['revenue'].min()
receita_maxima = orders['revenue'].max()

print(f' Os usuários não compraram menos que ${receita_minima}, \n e nem compraram mais que ${receita_maxima} por pedido')

 Os usuários não compraram menos que $5.0, 
 e nem compraram mais que $19920.4 por pedido


A priori, tudo ok aqui também: nenhuma receita negativa. Vamos analisar essas receitas extremas mais adiante no trabalho.

# 3. Priorizando Hipoteses

## 3.1. Aplicando o framework ICE para a escolha de hipóteses:

Vale lembrar que:

$ICE = (impact \ \ * \ \ confidence) \ / \ effort$

In [30]:
# Criando uma numeração para as hipóteses:
hipoteses = hipoteses.reset_index()

hipoteses['index'] = hipoteses['index'] + 1

hipoteses = hipoteses.rename(columns = {'index' : 'numeracao'})

hipoteses.head()

,numeracao,hypothesis,reach,impact,confidence,effort
0,1,Add two new channels for attracting traffic. T...,3,10,8,6
1,2,Launch your own delivery service. This will sh...,2,5,4,10
2,3,Add product recommendation blocks to the store...,8,3,7,3
3,4,Change the category structure. This will incre...,8,3,3,8
4,5,Change the background color on the main page. ...,3,1,1,1


In [31]:
# Calculando o ICE:
hipoteses['ICE'] = (hipoteses['impact'] * hipoteses['confidence']) / hipoteses['effort']

In [32]:
# Ordenando os dados pela coluna ICE em ordem decrescente:

hipoteses.sort_values(by = 'ICE', ascending = False)

,numeracao,hypothesis,reach,impact,confidence,effort,ICE
8,9,Launch a promotion that gives users discounts ...,1,9,9,5,16.200000
0,1,Add two new channels for attracting traffic. T...,3,10,8,6,13.333333
7,8,Add a subscription form to all the main pages....,10,7,8,5,11.200000
6,7,Show banners with current offers and sales on ...,5,3,8,3,8.000000
2,3,Add product recommendation blocks to the store...,8,3,7,3,7.000000
1,2,Launch your own delivery service. This will sh...,2,5,4,10,2.000000
5,6,Add a customer review page. This will increase...,3,2,2,3,1.333333
3,4,Change the category structure. This will incre...,8,3,3,8,1.125000
4,5,Change the background color on the main page. ...,3,1,1,1,1.000000


conclusões:

Olhando para o ICE, temos o seguinte pódio:

- Ouro para hipotese 9;
  
- Prata para hipotese 1;

- Bronze para a hipotese 8;

## 3.2. Aplicando o framework RICE para a escolha de hipóteses:

Vale lembrar que:

$RICE = (reach \ \ * \ \ impact \ \ * \ \ confidence) \ / \ effort$

In [33]:
# Calculando o RICE:
hipoteses['RICE'] = (hipoteses['reach'] * hipoteses['impact'] * hipoteses['confidence']) / hipoteses['effort']

In [34]:
# Ordenando os dados pela coluna RICE em ordem decrescente:

hipoteses.sort_values(by = 'RICE', ascending = False)

,numeracao,hypothesis,reach,impact,confidence,effort,ICE,RICE
7,8,Add a subscription form to all the main pages....,10,7,8,5,11.200000,112.0
2,3,Add product recommendation blocks to the store...,8,3,7,3,7.000000,56.0
0,1,Add two new channels for attracting traffic. T...,3,10,8,6,13.333333,40.0
6,7,Show banners with current offers and sales on ...,5,3,8,3,8.000000,40.0
8,9,Launch a promotion that gives users discounts ...,1,9,9,5,16.200000,16.2
3,4,Change the category structure. This will incre...,8,3,3,8,1.125000,9.0
1,2,Launch your own delivery service. This will sh...,2,5,4,10,2.000000,4.0
5,6,Add a customer review page. This will increase...,3,2,2,3,1.333333,4.0
4,5,Change the background color on the main page. ...,3,1,1,1,1.000000,3.0


conclusões:

Olhando para o RICE, temos o seguinte pódio:

- Ouro para hipotese 8;
  
- Prata para hipotese 3;

- Bronze para a hipotese 1;

## 3.3. Comparando os resultados e explicando as alterações:

conclusões:

Podemos afirmar que todas as alterações (no pódio e geral) se devem ao alcance (reach), uma vez que o RICE dar mais peso a hipoteses com maiores alcances de usuários.

Tivemos as seguintes alterações no pódio:

1. A Hipótese 8 subiu drasticamente
   
- ICE: 3º lugar → RICE: 1º lugar
- Por quê? Tem o maior alcance (reach = 10), então quando multiplicamos por alcance no RICE, ela dispara

2. A Hipótese 9 despencou
   
- ICE: 1º lugar → RICE: 5º lugar  
- Por quê? Tem alcance muito baixo (reach = 1), então mesmo com alto Impact e Confidence, afeta poucas pessoas

3. A Hipótese 3 subiu significativamente
   
- ICE: 5º lugar → RICE: 2º lugar
- Por quê? Tem alcance alto (reach = 8), compensando o Impact moderado

4. A hipotese 1 mostrou-se resiliente (permanecendo no pódio antes e após a inclusão do alcance), porém perdeu uma posição na alteração devido o seu alcance (reach = 3), o que é preocupante

- 2° lugar ICE → 3° lugar RICE

Diante disso, podemos concluir que:

- <b>Hipótese 8:</b> Adicionar um formulário de inscrição em todas as páginas principais - Busca capturar mais leads/contatos

- <b>Hipótese 3:</b> Adicionar blocos de recomendação de produtos na loja - Busca aumentar vendas através de sugestões personalizadas

podem ser boas opções, devido ter uma alta possibilidade de alcançar e impactar mais usuários.

# 4. Análise de teste A/B

## 4.1. Análises de receita e conversão (com gráficos)

### 4.1.1. Analisando a receita acumulada por grupo (A e B):

In [35]:
# Agrupando a receita por dia e grupo de teste e ordenando pelo dia:
revenue_per_day_group = orders.groupby(['date','group'])['revenue'].sum().reset_index().sort_values(by = 'date').reset_index(drop= True)

revenue_per_day_group.head()

C:\Users\Z4r4k1\AppData\Local\Temp\ipykernel_17540\1006478277.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  revenue_per_day_group = orders.groupby(['date','group'])['revenue'].sum().reset_index().sort_values(by = 'date').reset_index(drop= True)


,date,group,revenue
0,2019-08-01,A,2266.6
1,2019-08-01,B,967.2
2,2019-08-02,A,1468.3
3,2019-08-02,B,2568.1
4,2019-08-03,A,1815.2


In [36]:
# Filtrando revenue_per_day_group por grupo:
revenue_A = revenue_per_day_group[revenue_per_day_group['group'] == 'A'].copy() # Criando uma cópia para não 
                                                                                # correr riscos de alterar os arquivos originais
revenue_B = revenue_per_day_group[revenue_per_day_group['group'] == 'B'].copy()

In [37]:
# Calculando a receita diaria acumulada para cada grupo:
revenue_A['acumulada_A'] = revenue_A['revenue'].cumsum()
revenue_A = revenue_A[['date','acumulada_A']]

In [38]:
revenue_B['acumulada_B'] = revenue_B['revenue'].cumsum()
revenue_B = revenue_B[['date','acumulada_B']]

In [39]:
# unindo os dados:
receita_acumulada = revenue_A.merge(revenue_B, on = 'date')
receita_acumulada.head()

,date,acumulada_A,acumulada_B
0,2019-08-01,2266.6,967.2
1,2019-08-02,3734.9,3535.3
2,2019-08-03,5550.1,4606.9
3,2019-08-04,6225.6,6138.5
4,2019-08-05,7623.6,7587.8


In [40]:
# Plotando um gráfico de linhas para analisar as receitas acumuladas:
fig = px.line(receita_acumulada, 
              x='date', 
              y=['acumulada_A', 'acumulada_B'],
              title='Comparativo de Receitas Acumuladas diárias: Grupo A vs Grupo B',
              labels={'value': 'Receita Acumulada', 'date': 'Data', 'variable': 'Grupo'},
              markers=True)

newnames = {'acumulada_A': 'Grupo A', 'acumulada_B': 'Grupo B'}
fig.for_each_trace(lambda t: t.update(name = newnames.get(t.name, t.name)))

fig.update_layout(yaxis_tickprefix='$ ', hovermode='x unified')

fig.show()

conclusão:

- O Grupo A inicia com vantagem significativa ( 2.266 dólares vs  967 dólares no primeiro dia - 01/Ago), mas Grupo B acelera rapidamente e alcança paridade por volta do dia 2-3 de Agosto. A partir do meio do teste, ambos os grupos mantêm crescimento paralelo, sugerindo que a diferença inicial pode ter sido apenas variação natural do início do experimento.

- O Grupo de Teste (Grupo B) mostrou-se consistentemente acima do grupo de Controle (Grupo A) a partir do meio do teste: Mostrando que as mudanças impactaram positivamente a receita.

- **Implicação para o negócio:** A mudança testada demonstra capacidade de recuperação e superação da receita inicial, sugerindo potencial de **aumento sustentável da receita** se implementada permanentemente.

### 4.1.2. Analisando o tamanho médio acumulado do pedido por grupo (A e B):

Vale lembrar que:

$Tamanho \ Médio \ Acumulado \ do \ Pedido \ \ =  \ \ Receita \ Acumulada \ \ / \ \ Número \ de \ Pedidos \ Acumulado$

In [41]:
# Precisamos calcular o numero de pedidos acumulados por grupo:

# calculando o numero de pedidos diarios por grupo:
orders_per_day_group = orders.groupby(['date','group'])['transaction_id'].count().reset_index().sort_values(by = 'date').reset_index(drop= True)
orders_per_day_group.head()

# Filtrando os pedidos diárias por grupo:
orders_A = orders_per_day_group[orders_per_day_group['group'] == 'A'].copy()

orders_B = orders_per_day_group[orders_per_day_group['group'] == 'B'].copy()

# Calculando os pedidos acumulados por dia, para cada grupo:
orders_A['pedidos_A'] = orders_A['transaction_id'].cumsum()
orders_A = orders_A[['date','pedidos_A']]

orders_B['pedidos_B'] = orders_B['transaction_id'].cumsum()
orders_B = orders_B[['date','pedidos_B']]

# Juntando as informações:
pedidos_acumulados = orders_A.merge(orders_B, on='date')

# Juntando os pedidos acumulados e a receita acumulada:
pedidos_receitas_acumulados = pedidos_acumulados.merge(receita_acumulada, on='date')

# Calculando o tamanho médio acumulado do podido para cada grupo:
pedidos_receitas_acumulados['tamanho_medio_pedido_A'] = pedidos_receitas_acumulados['acumulada_A'] / pedidos_receitas_acumulados['pedidos_A']
pedidos_receitas_acumulados['tamanho_medio_pedido_B'] = pedidos_receitas_acumulados['acumulada_B'] / pedidos_receitas_acumulados['pedidos_B']

pedidos_receitas_acumulados.head()

C:\Users\Z4r4k1\AppData\Local\Temp\ipykernel_17540\3002504083.py:4: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



,date,pedidos_A,pedidos_B,acumulada_A,acumulada_B,tamanho_medio_pedido_A,tamanho_medio_pedido_B
0,2019-08-01,23,17,2266.6,967.2,98.547826,56.894118
1,2019-08-02,42,40,3734.9,3535.3,88.926190,88.382500
2,2019-08-03,66,54,5550.1,4606.9,84.092424,85.312963
3,2019-08-04,77,68,6225.6,6138.5,80.851948,90.272059
4,2019-08-05,99,89,7623.6,7587.8,77.006061,85.256180


In [42]:
# Plotando um gráfico de linhas para analisar o resultado:

fig = px.line(pedidos_receitas_acumulados,
              x='date',
              y=['tamanho_medio_pedido_A', 'tamanho_medio_pedido_B'],
              title='Tamanho médio acumulado do pedido por grupo: Grupo A vs Grupo B',
              labels={
                  'date': 'Data',
                  'value': 'Tamanho médio do pedido',
                  'variable': 'Grupo'
              },
              markers=True)

newnames = {'tamanho_medio_pedido_A': 'Grupo A', 'tamanho_medio_pedido_B': 'Grupo B'}
fig.for_each_trace(lambda t: t.update(name = newnames.get(t.name, t.name)))

fig.update_layout(yaxis_tickprefix='$ ', hovermode='x unified')

fig.show()

conclusão

- O Grupo A inicia com tamanho médio muito superior (98,55 vs 56,89 dólares no primeiro dia), mas ambos convergem rapidamente para valores similares (~$85-90) após poucos dias. A partir do meio do teste, Grupo B mantém ligeira vantagem, sugerindo que a mudança testada pode estar incentivando pedidos de valor ligeiramente maior.

- O grupo B Mantem-se a frente desde o meio do teste, mas essa diferença parece está caindo levemente.

- Isso quer dizer que essa vantagem pode está ligada a um valor atipico no ticket médio dos pedidos do dia 19, devido esse declinio gradual.

- **Implicação para o negócio:** A mudança pode estar incentivando **pedidos de maior valor**, mas a presença de possível valor atípico (dia 19) requer investigação antes da implementação definitiva.


### 4.1.3. Analisando a diferença relativa no tamanho médio acumulado do pedido para o grupo B em comparação com o grupo A:

Vale lembrar que:

$Diferenca \ Relativa \ \ = \ \ ( \ Métrica \ Acumulada \ B \ \ ÷ \ \ Métrica \ Acumulada \ A \ ) \ \ - \ \ 1$

No nosso caso fica:

$Diferenca \ Relativa \ \ = \ \ ( \ Tamanho \ medio \ acumulado \ do \ pedido \ B \ \ ÷ \ \ Tamanho \ medio \ acumulado \ do \ pedido \ B \ ) \ \ - \ \ 1$

In [43]:
# Calculando a diferença relativa percentual e convertendo para porcentagem:

pedidos_receitas_acumulados['diferenca_relativa_acumulada'] = 100 * ((pedidos_receitas_acumulados['tamanho_medio_pedido_B']
                                                               / pedidos_receitas_acumulados['tamanho_medio_pedido_A']) - 1)

pedidos_receitas_acumulados.head()

,date,pedidos_A,pedidos_B,acumulada_A,acumulada_B,tamanho_medio_pedido_A,tamanho_medio_pedido_B,diferenca_relativa_acumulada
0,2019-08-01,23,17,2266.6,967.2,98.547826,56.894118,-42.267506
1,2019-08-02,42,40,3734.9,3535.3,88.926190,88.382500,-0.611395
2,2019-08-03,66,54,5550.1,4606.9,84.092424,85.312963,1.451425
3,2019-08-04,77,68,6225.6,6138.5,80.851948,90.272059,11.651062
4,2019-08-05,99,89,7623.6,7587.8,77.006061,85.256180,10.713597


In [44]:
# Plotando um gráfico de linhas para visualizar a diferença relativa:

# 1. Criando a linha principal
fig = px.line(pedidos_receitas_acumulados,
              x='date',
              y='diferenca_relativa_acumulada',
              title='Diferença Relativa no tamanho médio acumulado do pedido (Grupo B com relação ao Grupo A)',
              labels={'date': 'Data', 'diferenca_relativa_acumulada': 'Diferença Relativa (em %)'},
              markers=True)

# 2. Adiciona a linha horizontal no zero (preta e tracejada)
fig.add_hline(y=0, line_dash="dash", line_color="black", line_width=2)

# 3. Adiciona sombreamento condicional
# Para positivo (Verde)
fig.add_scatter(x=pedidos_receitas_acumulados['date'], 
                y=pedidos_receitas_acumulados['diferenca_relativa_acumulada'].clip(lower=0),
                fill='tozeroy', mode='none', fillcolor='rgba(0, 255, 0, 0.2)', name='Positivo', showlegend=False)

# Para negativo (Vermelho)
fig.add_scatter(x=pedidos_receitas_acumulados['date'], 
                y=pedidos_receitas_acumulados['diferenca_relativa_acumulada'].clip(upper=0),
                fill='tozeroy', mode='none', fillcolor='rgba(255, 0, 0, 0.2)', name='Negativo', showlegend=False)

# Melhora o layout
fig.update_layout(hovermode='x unified')

fig.show()

conclusão

- Grupo B inicia com desvantagem severa (-42% no primeiro dia), mas recupera rapidamente e se torna superior a partir do dia 3, mantendo vantagem de ~10-15% em um certo momento. Isso pode ter ocorrido durante um periodo de adaptação.

- Porém, Existem oscilações nessa diferença, que embora mantenha-se positiva desde o meio do teste, oscilou 2 vezes entre positivo e negativo, e agora mantem-se com uma leve queda. Isso pode se dar por conta da presença de um valor atipico no dia 19, como mencionado anteriormente, pois esse dia marcou uma alta muito ascendente na diferença relativa entre B e A. Vamos verificar isso adiante.

- **Implicação para o negócio:** Após período de adaptação inicial, a mudança demonstra **impacto positivo consistente no valor médio dos pedidos**, justificando implementação com monitoramento de valores atípicos.


### 4.1.4. Analisando a taxa de conversão de cada grupo como a proporção de pedidos para o número de visitas para cada dia:

Vale lembrar que:

$conversao \ acumulada \ diaria \ \ = \ \ 100 \ * \ ( \ pedidos \ acumulados \ diarios \ \ / \ \ visitas \ acumuladas \ diarias \ )$

In [45]:
# Já temos os pedidos diarios acumulados para cada plano:
pedidos_acumulados = pedidos_receitas_acumulados[['date','pedidos_A','pedidos_B']].rename(
    columns = {'pedidos_A':'pedidos_acumulados_A','pedidos_B':'pedidos_acumulados_B'}
)
pedidos_acumulados.head()

,date,pedidos_acumulados_A,pedidos_acumulados_B
0,2019-08-01,23,17
1,2019-08-02,42,40
2,2019-08-03,66,54
3,2019-08-04,77,68
4,2019-08-05,99,89


In [46]:
# agora precisamos calcular as visitas acumuladas diárias para cada plano:

# filtrando as visitas diarias por grupo
visits_A = visits[visits['group'] == 'A'].copy()

visits_B = visits[visits['group'] == 'B'].copy()

# Calculando as visitas diarias acumuladas:
visits_A['visitas_acumuladas_A'] = visits_A['visits'].cumsum()
visits_A = visits_A[['date','visitas_acumuladas_A']]

visits_B['visitas_acumuladas_B'] = visits_B['visits'].cumsum()
visits_B = visits_B[['date','visitas_acumuladas_B']]

# Juntando as informações:
visitas_acumuladas = visits_A.merge(visits_B, on = 'date')

visitas_acumuladas.head()

,date,visitas_acumuladas_A,visitas_acumuladas_B
0,2019-08-01,719,713
1,2019-08-02,1338,1294
2,2019-08-03,1845,1803
3,2019-08-04,2562,2573
4,2019-08-05,3318,3280


In [47]:
# Vamos unir as informações e calcular a conversao acumulada:
funil_acumulado = visitas_acumuladas.merge(
    pedidos_acumulados,
    on = 'date'
)

funil_acumulado['conversao_acumulada_A'] = 100 * (funil_acumulado['pedidos_acumulados_A'] 
                                                  / funil_acumulado['visitas_acumuladas_A'])
funil_acumulado['conversao_acumulada_B'] = 100 * (funil_acumulado['pedidos_acumulados_B'] 
                                                  / funil_acumulado['visitas_acumuladas_B'])

funil_acumulado = funil_acumulado[['date','conversao_acumulada_A','conversao_acumulada_B']]

funil_acumulado.head()

,date,conversao_acumulada_A,conversao_acumulada_B
0,2019-08-01,3.198887,2.384292
1,2019-08-02,3.139013,3.091190
2,2019-08-03,3.577236,2.995008
3,2019-08-04,3.005464,2.642829
4,2019-08-05,2.983725,2.713415


In [48]:
# Visualizando o resultado com um gráfico de linhas:
fig = px.line(funil_acumulado,
              x='date',
              y=['conversao_acumulada_A', 'conversao_acumulada_B'],
              title='Taxa de conversão diária acumulada: Grupo A vs Grupo B',
              labels={
                  'date': 'Data',
                  'value': 'Taxa de conversão (em %)',
                  'variable': 'Grupo'
              },
              markers=True)

newnames = {'conversao_acumulada_A': 'Grupo A', 'conversao_acumulada_B': 'Grupo B'}
fig.for_each_trace(lambda t: t.update(name = newnames.get(t.name, t.name)))


fig.show()

conclusão:

- O Grupo A mantém conversão ligeiramente superior (3,2% vs 2,4% no início), mas ambos convergem para valores similares (2,7-3,0%) ao longo do teste. Recentemente o grupo B tomou aliderança em conversões, porém as oscilações são mínimas e não há diferença significativa entre os grupos, sugerindo que a mudança testada não impacta substancialmente a taxa de conversão dos visitantes.

- **Implicação para o negócio:** A mudança testada é **neutra em relação à conversão**, não prejudicando nem melhorando significativamente a capacidade de converter visitantes em compradores.

### 4.1.5. Analisando a diferença relativa na conversão cumulativa para o grupo B em comparação com o grupo A:

In [49]:
funil_acumulado['diferenca_relativa_conversao'] = 100 * ((funil_acumulado['conversao_acumulada_B']
                                                        / funil_acumulado['conversao_acumulada_A'])-1)


In [50]:
# Plotando um gráfico de linhas para visualizar a diferença relativa das conversões por grupo:

# 1. Criando a linha principal
fig = px.line(funil_acumulado,
              x='date',
              y='diferenca_relativa_conversao',
              title='Diferença Relativa entre as conversões acumuladas diárias: Grupo B com relação ao Grupo A',
              labels={'date': 'Data', 'diferenca_relativa_conversao': 'Diferença Relativa (em %)'},
              markers=True)

# 2. Adiciona a linha horizontal no zero (preta e tracejada)
fig.add_hline(y=0, line_dash="dash", line_color="black", line_width=2)

# 3. Adiciona sombreamento condicional
# Para positivo (Verde)
fig.add_scatter(x=funil_acumulado['date'], 
                y=funil_acumulado['diferenca_relativa_conversao'].clip(lower=0),
                fill='tozeroy', mode='none', fillcolor='rgba(0, 255, 0, 0.2)', name='Positivo', showlegend=False)

# Para negativo (Vermelho)
fig.add_scatter(x=funil_acumulado['date'], 
                y=funil_acumulado['diferenca_relativa_conversao'].clip(upper=0),
                fill='tozeroy', mode='none', fillcolor='rgba(255, 0, 0, 0.2)', name='Negativo', showlegend=False)

# Melhora o layout
fig.update_layout(hovermode='x unified')

fig.show()

conclusão:

- Com uma boa consistência, podemos dizer que atualmente o Grupo B (de teste) possui converte cerca de 15% a mais que o grupo A (de controle), essa diferença está estabilizada.

- O Grupo B inicia com desvantagem significativa (-25% no primeiro dia), mas rapidamente se recupera e estabiliza em vantagem positiva de ~15% a partir do meio do teste. A estabilização consistente em valores positivos indica que a mudança testada efetivamente melhora a taxa de conversão, com o Grupo B convertendo cerca de 15% mais visitantes que o Grupo A. Porém, essa diferença é só de 0.40%, como vimos anteriormente.

- **Implicação para o negócio:** A mudança demonstra **melhoria modesta mas estável na conversão** (15% relativo = 0,4% absoluto), que pode gerar impacto significativo em escala, considerando o volume de visitantes.


## 4.2 Identificação de anomalias

### 4.2.1. Calculando os percentis 95 e 99 para o número de pedidos por usuário, e definindo as anomalias:

In [51]:
# calculando o numero de pedidos de cada usuário:
orders_per_users = orders.groupby('visitor_id')['transaction_id'].count().reset_index().rename(columns = {'transaction_id':'transactions'})

In [52]:
# calculando os percentis 95 e 99:
percentis = np.percentile(orders_per_users['transactions'], [95,99])

percentil_95 = percentis[0]
percentil_99 = percentis[1]

print(f'Apenas menos de 5% dos usuários fizeram mais de {percentil_95} pedidos, \nenquanto menos de 1% fizeram mais de {percentil_99} pedidos')

Apenas menos de 5% dos usuários fizeram mais de 1.0 pedidos, 
enquanto menos de 1% fizeram mais de 2.0 pedidos


Então mais de 1 e 2 pedidos são nossos candidatos a possiveis valores atipicos. Vamos dar uma olhada mais de perto na dispersão dos dados para tomar essa escolha:

In [53]:
# Fazendo um gráfico de dispersão para pedidos:

orders_per_users = orders_per_users.reset_index().rename(columns={'index':'numero_do_usuario'})

orders_per_users['numero_do_usuario'] = orders_per_users['numero_do_usuario'] + 1

orders_per_users.head()

,numero_do_usuario,visitor_id,transactions
0,1,5114589,1
1,2,6958315,1
2,3,11685486,1
3,4,39475350,1
4,5,47206413,1


In [54]:
fig_corr = px.scatter(
    orders_per_users, 
    x='numero_do_usuario', 
    y='transactions',
    title='Distribuição de Pedidos por Usuário',
    labels={'numero_do_usuario': 'Numeração do Usuário (Índice)', 'transactions': 'Quantidade de Pedidos'},
    opacity=0.7,  # Ajuda a ver onde há mais densidade de pontos
    template='plotly_white'
)

fig_corr.update_layout(
    title_font_size=20,
    xaxis_showgrid=False,
    yaxis_gridcolor='LightGray'
)

fig_corr.show()

conclusão

95% dos usuários fizeram apenas 1 pedido. Dessa forma, vamos considerar o percentil_95 como limiar para demarcar nossos valores atípicos: Isto é, vamos considerar como atípico mais de 1 pedido por usuário

### 4.2.2 Fazendo um gráfico de dispersão dos preços dos pedidos:

In [55]:
pedidos = orders.reset_index().rename(columns={'index':'numero_do_pedido'})

pedidos['numero_do_pedido'] = pedidos['numero_do_pedido'] + 1

pedidos.head()

,numero_do_pedido,transaction_id,visitor_id,date,revenue,group
0,1,3667963787,3312258926,2019-08-15,30.4,B
1,2,2804400009,3642806036,2019-08-15,15.2,B
2,3,3797467345,1196621759,2019-08-15,155.1,B
3,4,2282983706,2322279887,2019-08-15,40.5,B
4,5,182168103,935554773,2019-08-15,35.0,B


In [56]:
fig_corr_2 = px.scatter(
    pedidos, 
    x='numero_do_pedido', 
    y='revenue',
    title='Distribuição da Receita por Pedido',
    labels={'numero_do_pedido': 'Numeração do Pedido (Índice)', 'revenue': 'Receita do Pedido (Em $)'},
    opacity=0.7,  # Ajuda a ver onde há mais densidade de pontos
    template='plotly_white'
)

fig_corr_2.update_layout(
    title_font_size=20,
    xaxis_showgrid=False,
    yaxis_gridcolor='LightGray'
)

fig_corr_2.show()

Nossa! Temos valores incrivelmente extremos: 3.120,10 e 19.920,40 dólares em um único pedido. Certamente, essa não é a realidade do dia a dia do negócio. Vamos dar uma olhada mais de perto:

In [57]:
fig_corr_2.update_yaxes(range=[0, 2000]) 
fig_corr_2.show()

Agora dar pra entender melhor a dispersão dos dados: Aparentemente, a faixa próxima a $500 mostra-se como um importante separador de valores atípicos para a receita por pedido. Vamos investigar isso mais de perto:

### 4.2.3 Calculando os percentis 95 e 99 dos preços dos pedidos, e definindo as anomalias:

In [58]:
# calculando os percentis 95 e 99:
percentis = np.percentile(pedidos['revenue'], [95,99])

percentil_95_revenue = percentis[0]
percentil_99_revenue = percentis[1]

print(f'Apenas menos de 5% dos pedidos retornaram mais de ${round(percentil_95_revenue,2)} em receita, \nenquanto menos de 1% retornaram mais de ${round(percentil_99_revenue,2)}')

Apenas menos de 5% dos pedidos retornaram mais de $414.28 em receita, 
enquanto menos de 1% retornaram mais de $830.3


conclusão:

Como suspeitado, menos de 5% dos pedidos retornaram mais de $414.28 em receita. Esse será nosso limiar para filtrar anomalias na receita por pedido.

## 4.3. Testes de significância estatística

### 4.3.1 Encontrando a significância estatística da diferença na conversão entre os grupos A e B usando os dados brutos:

Vamos aplicar o teste Z para duas proporções (Ele se aplica à veriáveis discretas não acumuladas):

- Hipótese Nula: A conversão não foi alterada no teste (conversão do grupo A <b>igual</b> à conversão do grupo B)
- Hipótese Alternativa: A conversão foi alterada no teste (conversão do grupo A <b>diferente</b> da conversão do grupo B)

In [59]:
# Vamos primeiro preparar os dados de conversão

# do Grupo A:
visitas_totais_A = visits[visits['group'] == 'A']['visits'].sum() # objeto do tipo 'float'
vendas_totais_A = orders[orders['group'] == 'A'].shape[0]         # objeto do tipo 'float'
conversao_A = vendas_totais_A / visitas_totais_A 

# do Grupo B:
visitas_totais_B = visits[visits['group'] == 'B']['visits'].sum() # objeto do tipo 'float'
vendas_totais_B = orders[orders['group'] == 'B'].shape[0]         # objeto do tipo 'float'
conversao_B = vendas_totais_B / visitas_totais_B

# Dados gerais:
conversao_geral = (vendas_totais_A + vendas_totais_B) / (visitas_totais_A + visitas_totais_B)

In [60]:
def estatistica_z(visitasA: 'float', visitasB: 'float', conversaoA: 'float', conversaoB: 'float', conversao_total: 'float', significancia: 'float' = 0.05):
    '''
    Essa função calcula a estatistica Z para a taxa de conversão de dois grupos quaisquer em um teste A/B (O teste é aplicado apenas
    para comparar variáveis discretas/não contínuas - Ex.: Valores absolutos, não acumulados discretos)

    PARAMETROS:
    visitasA: Visitas totais do grupo A -> 'float'
    visitasB: Visitas totais do grupo B -> 'float'
    conversaoA: Conversão total do grupo A -> 'float
    conversaoB: Conversão total do grupo B -> 'float
    conversao_total: Conversão total Geral -> 'float
    significancia: nivel de significância estatistica usado no teste - entre 0 e 1 (Definido 0.05 como padrão) -> 'float'

    RETURNS: -> f'str'
    '''
    
    # Erro padrão
    se = np.sqrt(conversao_total * (1 - conversao_total) * (1/visitasA + 1/visitasB))
    
    # Estatística Z
    z_stat = (conversaoB - conversaoA) / se
    
    # P-valor (teste bilateral)
    p_value = 2 * (1 - st.norm.cdf(abs(z_stat)))
    
    print(f"Taxa de conversão A: {100*conversaoA:.2f}%")
    print(f"Taxa de conversão B: {100*conversaoB:.2f}%")
    print(f"A diferença relativa na taxa de conversão é: {100*((conversaoB / conversaoA) - 1):.2f}%")
    print("")
    print(f"Nível de Significância usado: α={significancia}")
    print("")
    print(f"Estatística Z: {z_stat:.4f}")
    print(f"P-valor: {p_value:.4f}")
    print("")
    if p_value < significancia:
        print(f'❌Rejeitamos a Hipótese nula: A conversão foi alterada no teste (conversão do grupo A diferente da conversão do grupo B)')
    else: 
        print(f'✅Não Rejeitamos a Hipótese nula: A conversão não foi alterada no teste (conversão do grupo A igual à conversão do grupo B)')

In [61]:
# aplicando os testes no nosso conjunto de dados:
estatistica_z(visitas_totais_A, visitas_totais_B, conversao_A, conversao_B, conversao_geral)


Taxa de conversão A: 2.50%
Taxa de conversão B: 2.90%
A diferença relativa na taxa de conversão é: 15.98%

Nível de Significância usado: α=0.05

Estatística Z: 2.3899
P-valor: 0.0169

❌Rejeitamos a Hipótese nula: A conversão foi alterada no teste (conversão do grupo A diferente da conversão do grupo B)


Conclusão: 

<b>Insiders de negócio</b> - RESULTADO POSITIVO ✅:

Resultado estatístico: Diferença significativa (p-valor = 0.0169 < 0.05)

Impacto: Grupo B converte 2.90% vs 2.50% do Grupo A

Conclusão de negócio: A mudança testada aumenta a conversão em 0.40 pontos percentuais, representando um aumento relativo de 16% na taxa de conversão

### 4.3.2 Encontrando a significância estatística da diferença no tamanho médio do pedido entre os grupos A e B usando os dados brutos:

Vamos analisar a normalidade dos dados com o teste de Shapiro-Wilk e decidir por aplicar um dos dois testes para variáveis contínuas: T- test (ttest_ind) se normal, ou Mann-Whitney se não normal.

- Hipótese Nula: Os tamanhos médios dos pedidos dos grupos de teste são <b>iguais</b>.
- Hipótese Alternativa: Os tamanhos médios dos pedidos dos grupos de teste são <b>diferentes</b>.

In [62]:
# Preparando os dados:

revenue_A = orders[orders['group'] == 'A']['revenue']
revenue_B = orders[orders['group'] == 'B']['revenue']


In [63]:
def teste_dados_continuos(metrica_A, metrica_B, h_0: 'str', h_1: 'str', significancia: 'float' = 0.05):
    '''
    Essa função analisa a normalidade dos dados com o teste de Shapiro-Wilk e decide por aplicar um dos dois testes 
    para comparar a MÉDIA de duas métricas armazenadas em variáveis contínuas: T- test (ttest_ind) se normal
    , ou Mann-Whitney se não normal. Em seguida, aplica o teste e retorna o resultado.

    PARÂMETROS:
    metrica_A: Métrica contínua usada no grupo A (Ex.: Média, Valor bruto, Valor acumulado, etc...) -> DataFrame
    metrica_B: Métrica contínua usada no grupo B (Ex.: Média, Valor bruto, Valor acumulado, etc...) -> DataFrame
    h_0: Hipótese nula -> 'str'
    h_1: Hipótese alternativa -> 'str'
    significancia: Nivel de significancia estatistica usado - entre 0 e 1 (Definido 0.05 como padrão) -> 'float'

    RETURNS -> f'str'
    '''
    # 1. Testa a normalidade (Shapiro)
    shapiro_A = st.shapiro(metrica_A)
    shapiro_B = st.shapiro(metrica_B)
    
    print("Teste de Normalidade (Shapiro-Wilk):")
    print(f"Grupo A: p-valor = {shapiro_A.pvalue:.5f}")
    print(f"Grupo B: p-valor = {shapiro_B.pvalue:.5f}")

    print(' ') 
    print(f'Nivel de Significância usado para o Teste Shapiro: α={significancia}')
    print(' ')
    
    # 2. Decide qual teste usar
    if (shapiro_A.pvalue > significancia) and (shapiro_B.pvalue > significancia):
        # Dados normais → Teste t
        t_stat, p_value = st.ttest_ind(metrica_A, metrica_B)
        print("Usando teste t (dados normais)")
    else:
        # Dados não normais → Mann-Whitney
        u_stat, p_value = st.mannwhitneyu(metrica_A, metrica_B, alternative='two-sided')
        print("Usando Mann-Whitney (dados não normais)")
    
    print(' ')
    
    # 3. Executa o teste selecionado:
    print(f"P-valor: {p_value:.4f}")
    print(f'Nivel de Significância usado para o teste selecionado: α={significancia}')
    print(' ')
    
    if p_value < significancia:
            print(f'❌Rejeitamos a Hipótese nula: {h_1}')
    else: 
            print(f'✅Não Rejeitamos a Hipótese nula: {h_0}')


In [64]:
# Aplicando a função para analisar a diferença no tamanho médio do pedido:

hipotese_nula = 'Os tamanhos médios dos pedidos dos grupos de teste são IGUAIS.'
hipotese_alternativa = 'Os tamanhos médios dos pedidos dos grupos de teste são DIFERENTES.'

teste_dados_continuos(revenue_A, revenue_B, hipotese_nula, hipotese_alternativa)

Teste de Normalidade (Shapiro-Wilk):
Grupo A: p-valor = 0.00000
Grupo B: p-valor = 0.00000
 
Nivel de Significância usado para o Teste Shapiro: α=0.05
 
Usando Mann-Whitney (dados não normais)
 
P-valor: 0.8622
Nivel de Significância usado para o teste selecionado: α=0.05
 
✅Não Rejeitamos a Hipótese nula: Os tamanhos médios dos pedidos dos grupos de teste são IGUAIS.


In [65]:
print(f'O tamanho médio dos pedidos do Grupo A é: ${round(revenue_A.mean(),2)}')
print(f'O tamanho médio dos pedidos do Grupo B é: ${round(revenue_B.mean(),2)}')

O tamanho médio dos pedidos do Grupo A é: $113.7
O tamanho médio dos pedidos do Grupo B é: $145.35


conclusão:

<b>Insiders de negócios:</b> - SEM DIFERENÇA SIGNIFICATIVA ❌ no Tamanho Médio do Pedido:
  
Resultado estatístico: Não há diferença significativa (p-valor = 0.8622 > 0.05)

Valores: Grupo A = 113.70 dólares vs Grupo B = 145.35 dólares

Conclusão de negócio: Apesar do Grupo B ter valor médio maior, a diferença não é estatisticamente confiável para tomar decisões

### 4.3.3 Encontrando a significância estatística da diferença na conversão entre os grupos A e B usando os dados filtrados:

- Primeiro vamos filtrar os dados. Precisamos descartar as anomalias no número de pedidos para calcular a nova conversão, já que ela está correlacionada com o numéro de pedidos e visitas: Vamos detectar os ID's daqueles usuários que fizeram mais de 1 pedido e descarta-los do dataframe 'orders':

In [66]:
users_anormais = orders_per_users[orders_per_users['transactions'] > 1]['visitor_id']

orders_per_users[orders_per_users['transactions'] > 1].head(3) # todos esses IDs são de usuarios com mais de 1 pedido

,numero_do_usuario,visitor_id,transactions
55,56,249864742,3
82,83,366673373,2
94,95,406208401,2


In [67]:
mask_anomalias_1 = ~(orders['visitor_id'].isin(users_anormais))

filtered_orders = orders[mask_anomalias_1].reset_index(drop=True)

filtered_orders.head() # Esses são os pedidos de usuarios com exatamente 1 pedido

,transaction_id,visitor_id,date,revenue,group
0,3667963787,3312258926,2019-08-15,30.4,B
1,2804400009,3642806036,2019-08-15,15.2,B
2,3797467345,1196621759,2019-08-15,155.1,B
3,2282983706,2322279887,2019-08-15,40.5,B
4,182168103,935554773,2019-08-15,35.0,B


- Agora, vamos calcular a significância estatística:

In [68]:
# Calculando a conversão por grupo, para os dados filtrados

# do Grupo A:
visitas_A = visits[visits['group'] == 'A']['visits'].sum()                            # objeto do tipo 'float'
vendas_A_filtered = filtered_orders[filtered_orders['group'] == 'A'].shape[0]         # objeto do tipo 'float'
conversao_A_filtered = vendas_A_filtered / visitas_A 

# do Grupo B:
visitas_B = visits[visits['group'] == 'B']['visits'].sum()                            # objeto do tipo 'float'
vendas_B_filtered = filtered_orders[filtered_orders['group'] == 'B'].shape[0]         # objeto do tipo 'float'
conversao_B_filtered = vendas_B_filtered / visitas_B 

# Dados gerais:
conversao_geral_filtered = (vendas_A_filtered + vendas_B_filtered) / (visitas_A + visitas_B)

In [69]:
# Aplicando mais uma vez o teste Z, mas nos dados filtrados dessa vez:
estatistica_z(visitas_A, visitas_B, conversao_A_filtered, conversao_B_filtered, conversao_geral_filtered)


Taxa de conversão A: 2.28%
Taxa de conversão B: 2.70%
A diferença relativa na taxa de conversão é: 18.30%

Nível de Significância usado: α=0.05

Estatística Z: 2.5977
P-valor: 0.0094

❌Rejeitamos a Hipótese nula: A conversão foi alterada no teste (conversão do grupo A diferente da conversão do grupo B)


conclusão:

- Antes da filtragem: Diferença de +15.98%
  
- Após filtragem: Diferença de +18.30%

  
- Conclusão: A melhoria na conversão NÃO foi inflacionada por usuários super ativos (que fizeram múltiplas compras)

  
- A diferença relativa aumentou de 15.98% para 18.30%

  
- O p-valor melhorou de 0.0169 para 0.0094 (mais significativo)

  
- Interpretação: A mudança testada beneficia especialmente usuários "normais" (que fazem 1 compra)


### 4.3.4 Encontrando a significância estatística da diferença no tamanho médio do pedido entre os grupos usando os dados filtrados:

- Primeiro vamos filtrar os dados. Precisamos descartar as anomalias nos preços dos pedidos: Descartar aqueles pedidos que retornaram mais de $414.28

In [70]:
mask_anomalias_2 = orders['revenue'] <= percentil_95_revenue

In [71]:
filtered_orders = orders[mask_anomalias_2] # Filtrando apenas dados com receita menores ou iguais a $414.28 

filtered_orders.head()

,transaction_id,visitor_id,date,revenue,group
0,3667963787,3312258926,2019-08-15,30.4,B
1,2804400009,3642806036,2019-08-15,15.2,B
2,3797467345,1196621759,2019-08-15,155.1,B
3,2282983706,2322279887,2019-08-15,40.5,B
4,182168103,935554773,2019-08-15,35.0,B


- Agora, vamos calcular a significância estatística:

In [72]:
# Preparando os dados:

revenue_A_filtered = filtered_orders[filtered_orders['group'] == 'A']['revenue']
revenue_B_filtered = filtered_orders[filtered_orders['group'] == 'B']['revenue']


In [73]:
# Aplicando a função para analisar a diferença no tamanho médio do pedido:

hipotese_nula = 'Os tamanhos médios dos pedidos dos grupos de teste, após removidas as anomalias, são IGUAIS.'
hipotese_alternativa = 'Os tamanhos médios dos pedidos dos grupos de teste, após removidas as anomalias, são DIFERENTES.'

teste_dados_continuos(revenue_A_filtered, revenue_B_filtered, hipotese_nula, hipotese_alternativa)

Teste de Normalidade (Shapiro-Wilk):
Grupo A: p-valor = 0.00000
Grupo B: p-valor = 0.00000
 
Nivel de Significância usado para o Teste Shapiro: α=0.05
 
Usando Mann-Whitney (dados não normais)
 
P-valor: 0.7324
Nivel de Significância usado para o teste selecionado: α=0.05
 
✅Não Rejeitamos a Hipótese nula: Os tamanhos médios dos pedidos dos grupos de teste, após removidas as anomalias, são IGUAIS.


In [74]:
print(f'O tamanho médio dos pedidos do Grupo A, após removidas as anomalias, é: ${round(revenue_A_filtered.mean(),2)}')
print(f'O tamanho médio dos pedidos do Grupo B, após removidas as anomalias, é: ${round(revenue_B_filtered.mean(),2)}')


O tamanho médio dos pedidos do Grupo A, após removidas as anomalias, é: $83.09
O tamanho médio dos pedidos do Grupo B, após removidas as anomalias, é: $78.33


conclusão: 

- Antes da filtragem: Grupo B = 145.35 dólares vs Grupo A = 113.70 dólares (diferença aparente)
  
- Após filtragem: Grupo B = 78.33 dólares vs Grupo A = 83.09 dólares (sem diferença significativa)

<b>O que isso revela:</b>

Valores atípicos estavam inflacionando os resultados

Sem anomalias: Não há diferença real no ticket médio

A mudança testada é neutra em relação ao valor dos pedidos

📈 <b>Impacto para o Negócio:</b>

EXCELENTE NOTÍCIA! 

A melhoria na conversão (seção 4.3.3) não vem às custas do ticket médio

A mudança aumenta conversões sem prejudicar o valor médio dos pedidos

Resultado líquido: Mais vendas com ticket médio mantido = mais receita total

# 5. Conclusões Finais: Tomando uma decisão com base nos resultados do teste

Já temos um veredito final:

🏆 VITÓRIA DO GRUPO B.

Devemos **ENCERRAR O TESTE e implementar a VERSÃO B em 100% do tráfego**, devido às seguintes evidências coletadas durante a análise:

### 1) Conversão (Dados Filtrados - 4.3.3):
- P-valor = 0.0094 (altamente significativo)
- +18.30% de melhoria relativa
- Resultado mais forte após remoção de anomalias
### 2) Ticket Médio (Dados Filtrados - 4.3.4):
- P-valor = 0.7324 (sem diferença significativa)
- Impacto neutro 
- não prejudica o valor médio dos pedidos
### 3) Evidências dos Gráficos (4.1):
- Receita acumulada: Grupo B supera consistentemente
- Conversão: +15% relativo estabilizado
- Sustentabilidade: Resultados mantidos ao longo do tempo
### 4) Validação da Robustez:
- Dados brutos vs filtrados: Resultado se fortaleceu após filtragem
- Anomalias removidas: Confirmou que o efeito é genuíno
- Consistência temporal: Vantagem sustentada do Grupo B
### 5) Impacto de Negócio:
**IMPLEMENTAR GRUPO B**, porque:

✅ Aumenta conversões (+0.42 pontos percentuais)

✅ Mantém ticket médio (neutro)

✅ Resultado estatisticamente confiável (p < 0.01)

✅ Efeito robusto (livre de anomalias)